In [4]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('/workspace/VTCM_PYTHON')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from VTCM_solver import *
from PINO.dataset import *
from PINO.pino_utils import VTCMHDF5MapStyleDataset, create_vtcm_hdf5_dataloader

full_seq_h5 = '/workspace/VTCM_PYTHON/datasets/VTCM_vertical/train_full_seq.hdf5'
ds = VTCMHDF5MapStyleDataset(full_seq_h5, device='cpu', preload_to_memory=False)
sample = ds[0]
print(type(ds), len(ds), sample.keys())

<class 'PINO.pino_utils.VTCMHDF5MapStyleDataset'> 160 dict_keys(['input', 'output', 'dt', 'source_file', 'vehicle_params', 'rail_params', 'fastener_params', 'subrail_params', 'line_params', 'vx_mps'])


In [21]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('/workspace/VTCM_PYTHON')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from VTCM_solver import *
vehicle_params = sample['vehicle_params']
print(vehicle_params)
m_c = vehicle_params[0]
m_f = vehicle_params[1]
m_r = vehicle_params[1]
k_pz = vehicle_params[18]
c_pz = vehicle_params[21]
k_sz = vehicle_params[24]
c_sz = vehicle_params[27]
g = 9.81
lc = vehicle_params[34]
solver = VTCMCoupledDynamics(
    m_c=m_c,
    m_f=m_f,
    m_r=m_r,
    k_pz=k_pz,
    c_pz=c_pz,
    k_sz=k_sz,
    c_sz=c_sz,
    g=g,
    lc=lc,
    time=True
)
solver.pprint()

tensor([ 3.4000e+04,  3.0000e+03,  1.4000e+03,  0.0000e+00,  7.5060e+04,
         2.2770e+06,  2.0860e+06,  2.2600e+03,  2.7100e+03,  3.1600e+03,
         9.1500e+02,  1.4000e+02,  9.1500e+02,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  1.0000e+07,  5.0000e+06,  5.5000e+05,  0.0000e+00,
         0.0000e+00,  6.0000e+03,  1.5000e+05,  1.5000e+05,  4.0000e+05,
         0.0000e+00,  6.0000e+04,  8.0000e+04,  0.0000e+00,  1.4150e+00,
        -8.1000e-02,  1.4000e-01,  9.7800e-01,  9.7800e-01,  9.0000e+00,
         1.2000e+00,  4.5750e-01,  2.5000e+06,  4.2232e-08,  5.5917e+04])
dynamic_carbody: 34000.0*a_c(x, t) + 160000.0*v_c(x, t) - 80000.0*v_f(x, t) - 80000.0*v_r(x, t) + 800000.0*x_c(x, t) - 400000.0*x_f(x, t) - 400000.0*x_r(x, t)
dynamic_front_bogie: 3000.0*a_f(x, t) + 3600000.0*pitch_c(x, t) - 80000.0*v_c(x, t) + 92000.0*v_f(x, t) + 720000.0*v_pitch_c(x, t) - 6000.0*v_w1(x, t) - 6000.0*v_w2(x, t) - 400000.0*x_c(x, t) + 1500000.0*x_f(x, t) - 550000.0*x_w1(x, t) - 550000.0*x_w2(x, t

In [22]:
# 3) 先反归一化，再把 sample 的单时刻数值代入 solver.equations
import sympy as sp
import numpy as np

# 建议：参数先转 float，再构建 solver（避免 torch.Tensor 参与符号运算）
def to_float(x):
    return float(x.detach().cpu().item()) if hasattr(x, 'detach') else float(x)

solver_num = VTCMCoupledDynamics(
    m_c=to_float(vehicle_params[0]),
    m_f=to_float(vehicle_params[1]),
    m_r=to_float(vehicle_params[1]),
    k_pz=to_float(vehicle_params[18]),
    c_pz=to_float(vehicle_params[21]),
    k_sz=to_float(vehicle_params[24]),
    c_sz=to_float(vehicle_params[27]),
    g=9.81,
    lc=to_float(vehicle_params[34]),
    time=True,
)

# 输出通道约定（当前数据集）:
# disp: [car, front_bogie, rear_bogie, w1, w2, w3, w4] -> 0..6
# vel : 同顺序 -> 7..13
# acc : 同顺序 -> 14..20
y_norm = sample['output'].detach().cpu().numpy()  # [21, T] 归一化后

# === 先反归一化 ===
norm_path = '/workspace/VTCM_PYTHON/datasets/VTCM_vertical/norm_stats.npz'
st = np.load(norm_path)
X_mean = st['X_mean'].reshape(-1, 1)  # [21,1]
X_std = st['X_std'].reshape(-1, 1)    # [21,1]

if y_norm.shape[0] != X_mean.shape[0]:
    raise ValueError(f'output通道数与norm_stats不一致: y={y_norm.shape[0]}, X_mean={X_mean.shape[0]}')

y = y_norm * X_std + X_mean  # 反归一化后的 [21, T]

k = 10000  # 选择一个时刻索引
x_c_val, x_f_val, x_r_val = y[0, k], y[1, k], y[2, k]
v_c_val, v_f_val, v_r_val = y[7, k], y[8, k], y[9, k]
a_c_val, a_f_val, a_r_val = y[14, k], y[15, k], y[16, k]

x_w1_val, x_w2_val, x_w3_val, x_w4_val = y[3, k], y[4, k], y[5, k], y[6, k]
v_w1_val, v_w2_val, v_w3_val, v_w4_val = y[10, k], y[11, k], y[12, k], y[13, k]

# 你当前 21 通道里没有 pitch_c / v_pitch_c，这里先按 0 代入
pitch_c_val = 0.0
v_pitch_c_val = 0.0

x, t = sp.Symbol('x'), sp.Symbol('t')
subs_map = {
    sp.Function('x_c')(x, t): x_c_val,
    sp.Function('v_c')(x, t): v_c_val,
    sp.Function('a_c')(x, t): a_c_val,
    sp.Function('x_f')(x, t): x_f_val,
    sp.Function('v_f')(x, t): v_f_val,
    sp.Function('a_f')(x, t): a_f_val,
    sp.Function('x_r')(x, t): x_r_val,
    sp.Function('v_r')(x, t): v_r_val,
    sp.Function('a_r')(x, t): a_r_val,
    sp.Function('x_w1')(x, t): x_w1_val,
    sp.Function('v_w1')(x, t): v_w1_val,
    sp.Function('x_w2')(x, t): x_w2_val,
    sp.Function('v_w2')(x, t): v_w2_val,
    sp.Function('x_w3')(x, t): x_w3_val,
    sp.Function('v_w3')(x, t): v_w3_val,
    sp.Function('x_w4')(x, t): x_w4_val,
    sp.Function('v_w4')(x, t): v_w4_val,
    sp.Function('pitch_c')(x, t): pitch_c_val,
    sp.Function('v_pitch_c')(x, t): v_pitch_c_val,
}

print(f'time index k={k} (denormalized)')
for name, eq in solver_num.equations.items():
    val = float(sp.N(eq.subs(subs_map)))
    print(f'{name:>24s} residual = {val:.6e}')

time index k=10000 (denormalized)
         dynamic_carbody residual = -7.445464e+01
     dynamic_front_bogie residual = 4.412963e+00
      dynamic_rear_bogie residual = -4.997534e+00


In [23]:
# 4) 数值诊断：看残差是否是“量纲大但相对误差不大”
import numpy as np
import sympy as sp

x, t = sp.Symbol('x'), sp.Symbol('t')

def build_subs_at_k(y, k):
    x_c_val, x_f_val, x_r_val = y[0, k], y[1, k], y[2, k]
    v_c_val, v_f_val, v_r_val = y[7, k], y[8, k], y[9, k]
    a_c_val, a_f_val, a_r_val = y[14, k], y[15, k], y[16, k]
    x_w1_val, x_w2_val, x_w3_val, x_w4_val = y[3, k], y[4, k], y[5, k], y[6, k]
    v_w1_val, v_w2_val, v_w3_val, v_w4_val = y[10, k], y[11, k], y[12, k], y[13, k]

    return {
        sp.Function('x_c')(x, t): x_c_val,
        sp.Function('v_c')(x, t): v_c_val,
        sp.Function('a_c')(x, t): a_c_val,
        sp.Function('x_f')(x, t): x_f_val,
        sp.Function('v_f')(x, t): v_f_val,
        sp.Function('a_f')(x, t): a_f_val,
        sp.Function('x_r')(x, t): x_r_val,
        sp.Function('v_r')(x, t): v_r_val,
        sp.Function('a_r')(x, t): a_r_val,
        sp.Function('x_w1')(x, t): x_w1_val,
        sp.Function('v_w1')(x, t): v_w1_val,
        sp.Function('x_w2')(x, t): x_w2_val,
        sp.Function('v_w2')(x, t): v_w2_val,
        sp.Function('x_w3')(x, t): x_w3_val,
        sp.Function('v_w3')(x, t): v_w3_val,
        sp.Function('x_w4')(x, t): x_w4_val,
        sp.Function('v_w4')(x, t): v_w4_val,
        sp.Function('pitch_c')(x, t): 0.0,
        sp.Function('v_pitch_c')(x, t): 0.0,
    }

def eq_diagnostics(eq, subs):
    terms = eq.as_ordered_terms()
    vals = np.array([float(sp.N(term.subs(subs))) for term in terms], dtype=float)
    residual = vals.sum()
    abs_sum = np.abs(vals).sum() + 1e-12
    rel_balance = np.abs(residual) / abs_sum
    return residual, abs_sum, rel_balance, vals

# 单点分解（默认 k）
subs_k = build_subs_at_k(y, k)
print(f'---- 单点分解 k={k} ----')
for name, eq in solver_num.equations.items():
    residual, abs_sum, rel_balance, vals = eq_diagnostics(eq, subs_k)
    print(f'{name:>24s}: residual={residual:+.6e}, abs_sum={abs_sum:.6e}, |res|/sum|term|={rel_balance:.6e}')

# 全序列抽样统计
T = y.shape[1]
num_probe = min(300, T)
idx = np.linspace(0, T - 1, num_probe, dtype=int)
print(f'\n---- 抽样统计 N={len(idx)} ----')
for name, eq in solver_num.equations.items():
    rels, abss, rss = [], [], []
    for kk in idx:
        subs = build_subs_at_k(y, kk)
        residual, abs_sum, rel_balance, _ = eq_diagnostics(eq, subs)
        rels.append(rel_balance)
        abss.append(abs(residual))
        rss.append(residual)
    rels = np.array(rels)
    abss = np.array(abss)
    rss = np.array(rss)
    print(f'{name:>24s}: med(|res|)={np.median(abss):.3e}, p95(|res|)={np.percentile(abss,95):.3e}, med(rel)={np.median(rels):.3e}, p95(rel)={np.percentile(rels,95):.3e}, mean(res)={np.mean(rss):+.3e}')

---- 单点分解 k=10000 ----
         dynamic_carbody: residual=-7.445464e+01, abs_sum=1.168383e+03, |res|/sum|term|=6.372451e-02
     dynamic_front_bogie: residual=+4.412963e+00, abs_sum=9.045871e+02, |res|/sum|term|=4.878428e-03
      dynamic_rear_bogie: residual=-4.997534e+00, abs_sum=1.823244e+03, |res|/sum|term|=2.741012e-03

---- 抽样统计 N=300 ----
         dynamic_carbody: med(|res|)=2.473e+02, p95(|res|)=8.946e+02, med(rel)=1.043e-01, p95(rel)=2.580e-01, mean(res)=-1.343e+01
     dynamic_front_bogie: med(|res|)=1.111e+02, p95(|res|)=4.988e+02, med(rel)=3.932e-02, p95(rel)=1.692e-01, mean(res)=-1.298e-02
      dynamic_rear_bogie: med(|res|)=9.367e+01, p95(|res|)=4.545e+02, med(rel)=3.522e-02, p95(rel)=1.613e-01, mean(res)=-3.171e+00


In [19]:
# 5) 检查反归一化后的量级（是否接近“动态增量 around 0”）
names = [
    'x_c','x_f','x_r','x_w1','x_w2','x_w3','x_w4',
    'v_c','v_f','v_r','v_w1','v_w2','v_w3','v_w4',
    'a_c','a_f','a_r','a_w1','a_w2','a_w3','a_w4'
 ]

print('channel stats: mean / std / min / max')
for i, n in enumerate(names):
    arr = y[i]
    print(f'{i:02d} {n:>4s}: mean={arr.mean():+.4e}, std={arr.std():.4e}, min={arr.min():+.4e}, max={arr.max():+.4e}')

print('\nmg terms:')
print(f'm_c*g = {to_float(vehicle_params[0])*9.81:.3f}')
print(f'm_f*g = {to_float(vehicle_params[1])*9.81:.3f}')
print(f'm_r*g = {to_float(vehicle_params[1])*9.81:.3f}')

channel stats: mean / std / min / max
00  x_c: mean=+4.4490e-04, std=6.8065e-04, min=-7.3019e-04, max=+1.8897e-03
01  x_f: mean=+4.5252e-04, std=6.8956e-04, min=-1.0000e-03, max=+1.9767e-03
02  x_r: mean=+4.3422e-04, std=6.4082e-04, min=-1.0172e-03, max=+1.9598e-03
03 x_w1: mean=+4.2811e-04, std=7.6865e-04, min=-1.7780e-03, max=+2.3100e-03
04 x_w2: mean=+4.5858e-04, std=7.6754e-04, min=-1.7478e-03, max=+2.3782e-03
05 x_w3: mean=+4.3813e-04, std=7.5556e-04, min=-1.7601e-03, max=+2.3418e-03
06 x_w4: mean=+4.2178e-04, std=7.5343e-04, min=-1.7830e-03, max=+2.3657e-03
07  v_c: mean=+1.8887e-04, std=2.6552e-03, min=-7.0726e-03, max=+8.2705e-03
08  v_f: mean=+2.5656e-05, std=4.6574e-03, min=-1.3848e-02, max=+1.3530e-02
09  v_r: mean=+2.0167e-04, std=4.1917e-03, min=-1.3125e-02, max=+1.1765e-02
10 v_w1: mean=+7.6365e-05, std=2.3261e-02, min=-7.5569e-02, max=+8.8952e-02
11 v_w2: mean=+4.4494e-05, std=2.3295e-02, min=-8.3046e-02, max=+8.0066e-02
12 v_w3: mean=+7.9491e-05, std=2.1879e-02, min=-7.

In [20]:
# 6) 重新加载 VTCM_solver，确保使用最新方程
import importlib
import VTCM_solver as vtcm_solver_module
importlib.reload(vtcm_solver_module)
from VTCM_solver import *

In [ ]:
import torch.nn as nn
class PINO(nn.Module):
     
    

